# Extra Trees — Tokke–Vinje

This standalone notebook reads raw competition data. It trains 14 Extra Trees classifiers, one per generator, with the same inputs, validation folds and metric as `logistic_regression_baseline.ipynb`. The results are directly comparable. Validation uses expanding windows for 2020, 2021 and 2022 with a seven-day gap. All probabilities are scored together with micro ROC-AUC.

Run all cells from the repository root or `notebooks/`. Predictions and feature importances go to `outputs/extra_trees/`. Scores are merged into the shared files in `outputs/evaluation/`; rows from other models are kept. The same input assumptions apply: no future reservoir volumes or calculation times are used. Full-horizon prices, inflows and terminal water values are assumed to be allowed scheduling inputs.

Hyperparameters are scikit-learn defaults, not tuned results: 100 trees, `max_features="sqrt"`, `min_samples_leaf=1`, no bootstrap. Only `n_jobs=4` (the course's "normal PC") and `random_state` are set. The feature set is identical to logistic regression, including its interaction columns, and there is no feature selection. Trees use the calendar variables as ordinal numbers, so no scaling or one-hot encoding is needed. No class balancing is applied.

In [1]:
import os
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[name] = "4"
import json, re, time
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import yaml
import sklearn
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import roc_auc_score

SEED = 71
H = 168
VALIDATION_YEARS = (2020, 2021, 2022)
MODEL = "extra_trees"
ET_PARAMS = {"n_estimators": 100, "max_features": "sqrt", "min_samples_leaf": 1, "bootstrap": False}
ROOT = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / "data/kernel").is_dir())
OUT = ROOT / "outputs"
MODEL_DIR = OUT / MODEL
EVAL_DIR = OUT / "evaluation"
for directory in (MODEL_DIR, EVAL_DIR):
    directory.mkdir(parents=True, exist_ok=True)
KERNEL = ROOT / "data/kernel"
EXTENDED = ROOT / "data/extended"

## Load and validate raw data

In [2]:
def read_series(path):
    frame = pd.read_csv(path)
    frame["date"] = pd.to_datetime(frame["date"], utc=True)
    frame = frame.set_index("date").sort_index()
    assert frame.index.is_unique and not frame.isna().any().any(), path
    return frame

uc = pd.read_csv(KERNEL / "Unit_commitment_decisions.csv")
test_map = pd.read_csv(ROOT / "prediction_mapping.csv")
sample = pd.read_csv(ROOT / "sample_submission.csv")
price_raw = read_series(KERNEL / "Historical_day_ahead_price_2015_2025.csv")
inflow = read_series(KERNEL / "Historical_inflow_1958_2025.csv")
volume = read_series(KERNEL / "Historical_volume_2015_2024.csv")
water_value = read_series(KERNEL / "Synthetic_water_value_2015_2024.csv")
with (EXTENDED / "Tokke_Vinje_topology.yaml").open() as handle:
    topology = yaml.safe_load(handle)
assert "Day-ahead price (EUR/MWh)" in price_raw
price = price_raw["Day-ahead price (EUR/MWh)"].resample("h").mean()
label_cols = [c for c in sample if c != "Run No"]
UNITS = list(dict.fromkeys(re.fullmatch(r"result_committed_(.+)_t\d+", c).group(1) for c in label_cols))
assert len(UNITS) == 14
assert label_cols == [f"result_committed_{u}_t{t}" for u in UNITS for t in range(H)]
uc = uc.sort_values("Run No").reset_index(drop=True)
cases = pd.concat([uc[["Run No", "starttime"]].assign(split="train"),
                   test_map.assign(split="test")], ignore_index=True).sort_values("Run No").reset_index(drop=True)
cases["starttime"] = pd.to_datetime(cases["starttime"], utc=True)
assert cases["Run No"].is_unique
assert set(test_map["Run No"]) == set(sample["Run No"])
Y = uc[label_cols].to_numpy(dtype=np.float32).reshape(len(uc), len(UNITS), H).transpose(0, 2, 1)
assert np.isin(Y, [0, 1]).all()
print(cases.groupby("split")["starttime"].agg(["min", "max", "count"]))
print("Training target shape:", Y.shape)


                            min                       max  count
split                                                           
test  2023-01-01 00:00:00+00:00 2024-12-25 00:00:00+00:00    725
train 2015-01-01 00:00:00+00:00 2022-12-31 00:00:00+00:00   2922
Training target shape: (2922, 168, 14)


## Build features from available scheduling inputs

Calendar days are computed directly from timestamps. Initial storage and terminal water values are looked up exactly. Historical price features are omitted to avoid fabricated history for early cases. Equal prices receive equal average ranks. Reservoir mappings are derived from YAML connections.

In [3]:
connections = pd.DataFrame(topology["connections"])
def feeding_reservoirs(plant):
    pending, seen, found = [plant], set(), set()
    while pending:
        node = pending.pop()
        if node in seen:
            continue
        seen.add(node)
        for _, link in connections.loc[connections["to"] == node].iterrows():
            if link["from_type"] == "reservoir":
                found.add(link["from"])
            elif link["from_type"] in ("river", "tunnel"):
                pending.append(link["from"])
    assert found, plant
    return sorted(found)

UNIT_RESERVOIRS = {u: feeding_reservoirs(u.rsplit("_", 1)[0]) for u in UNITS}
starts = pd.DatetimeIndex(cases["starttime"])
timestamps = starts.tz_convert(None).to_numpy()[:, None] + np.arange(H)[None, :] * np.timedelta64(1, "h")
flat_times = pd.DatetimeIndex(timestamps.ravel()).tz_localize("UTC")
local = flat_times.tz_convert("Europe/Oslo")
P = price.reindex(flat_times).to_numpy(dtype=np.float32).reshape(-1, H)
assert np.isfinite(P).all(), "Missing horizon prices; do not silently fill."
rank = pd.DataFrame(P).rank(axis=1, method="average", pct=True).to_numpy(dtype=np.float32)
X = pd.DataFrame({
    "t": np.tile(np.arange(H), len(cases)),
    "hour": local.hour,
    "weekday": local.dayofweek,
    "season": (local.month - 1) // 3,
    "doy_sin": np.sin(2 * np.pi * local.dayofyear / 365.25),
    "doy_cos": np.cos(2 * np.pi * local.dayofyear / 365.25),
    "price": P.ravel(),
    "price_rank": rank.ravel(),
    "price_minus_mean": (P - P.mean(axis=1, keepdims=True)).ravel(),
    "price_week_mean": np.repeat(P.mean(axis=1), H),
})
for unit, reservoirs in UNIT_RESERVOIRS.items():
    terminal = water_value.loc[starts + pd.Timedelta(days=7), reservoirs].mean(axis=1).to_numpy(dtype=np.float32)
    initial_fill = np.mean([volume.loc[starts, r].to_numpy() / topology["model"]["reservoir"][r]["max_vol"] for r in reservoirs], axis=0)
    unit_inflow = inflow.reindex(flat_times)[reservoirs].sum(axis=1, min_count=len(reservoirs)).to_numpy(dtype=np.float32)
    X[f"wv_{unit}"] = np.repeat(terminal, H)
    X[f"fill_{unit}"] = np.repeat(initial_fill, H)
    X[f"inflow_{unit}"] = unit_inflow
    X[f"spread_{unit}"] = X["price"] - X[f"wv_{unit}"]
    X[f"rank_fill_{unit}"] = X["price_rank"] * X[f"fill_{unit}"]
    X[f"spread_t_{unit}"] = X[f"spread_{unit}"] * X["t"] / (H - 1)
X = X.astype(np.float32)
assert np.isfinite(X.to_numpy()).all()
assert np.array_equal(cases.loc[cases["split"] == "train", "Run No"], uc["Run No"])
print("Feature shape:", X.shape, "Memory (MB):", X.memory_usage(deep=True).sum() / 1e6)


Feature shape: (612696, 94) Memory (MB): 230.373828


## Extra Trees and validation

Each generator gets its own forest on the same columns as logistic regression: shared calendar and price features plus that generator's water value, fill, inflow and interaction features. Each forest fits only on the current training fold. One forest holds roughly 300–400 MB, so it predicts and is then discarded instead of keeping all 14 in memory. Feature importance is impurity-based (MDI), computed on training data. It is biased towards continuous high-cardinality features and shows associations, not causal effects.

In [4]:
COMMON = ["t", "hour", "weekday", "season", "doy_sin", "doy_cos", "price", "price_rank", "price_minus_mean", "price_week_mean"]
def columns_for(unit):
    return COMMON + [f"{prefix}_{unit}" for prefix in ("wv", "fill", "inflow", "spread", "rank_fill", "spread_t")]

def micro_auc(truth, probability):
    assert truth.shape == probability.shape
    assert np.isfinite(probability).all() and ((probability >= 0) & (probability <= 1)).all()
    return float(roc_auc_score(truth.reshape(-1), probability.reshape(-1)))

def rows_for(case_indices):
    return (np.asarray(case_indices)[:, None] * H + np.arange(H)[None, :]).ravel()

def fit_predict_extra_trees(train_features, targets, predict_features):
    """Fit one forest per generator, predict immediately and keep only predictions and importances."""
    predictions, importances = np.empty((len(predict_features), len(UNITS)), np.float32), []
    for j, unit in enumerate(UNITS):
        columns = columns_for(unit)
        if np.unique(targets[:, j]).size < 2:
            predictions[:, j] = targets[:, j].mean()
            continue
        model = ExtraTreesClassifier(**ET_PARAMS, n_jobs=4, random_state=SEED)
        model.fit(train_features[columns], targets[:, j])
        predictions[:, j] = model.predict_proba(predict_features[columns])[:, 1]
        importances.append(pd.DataFrame({"unit": unit, "feature": columns, "importance": model.feature_importances_}))
        del model
    return predictions, pd.concat(importances, ignore_index=True)

results, importance_tables = [], []
for year in VALIDATION_YEARS:
    valid_start = pd.Timestamp(f"{year}-01-01", tz="UTC")
    train_indices = cases.index[(cases["split"] == "train") & (cases["starttime"] < valid_start - pd.Timedelta(days=7))].to_numpy()
    valid_indices = cases.index[(cases["split"] == "train") & (cases["starttime"].dt.year == year)].to_numpy()
    assert cases.loc[train_indices, "starttime"].max() + pd.Timedelta(days=7) < cases.loc[valid_indices, "starttime"].min()
    xt, xv = X.iloc[rows_for(train_indices)], X.iloc[rows_for(valid_indices)]
    yt, yv = Y[train_indices].reshape(-1, len(UNITS)), Y[valid_indices].reshape(-1, len(UNITS))
    started = time.monotonic()
    prediction, importance = fit_predict_extra_trees(xt, yt, xv)
    results.append({"model": MODEL, "validation_year": year, "micro_auc": micro_auc(yv, prediction),
                    "train_cases": len(train_indices), "valid_cases": len(valid_indices)})
    ordered = prediction.reshape(-1, H, len(UNITS)).transpose(0, 2, 1).reshape(len(valid_indices), -1)
    validation_frame = pd.DataFrame(ordered, columns=label_cols)
    validation_frame.insert(0, "Run No", cases.loc[valid_indices, "Run No"].to_numpy())
    validation_frame.to_csv(MODEL_DIR / f"oof_{MODEL}_{year}.csv", index=False)
    importance_tables.append(importance.assign(validation_year=year))
    print(year, results[-1], "Elapsed seconds:", round(time.monotonic() - started, 1), flush=True)

scores = pd.DataFrame(results)
print(scores.groupby("model")["micro_auc"].agg(["mean", "std", "min", "max"]))
importances = pd.concat(importance_tables, ignore_index=True)[["validation_year", "unit", "feature", "importance"]]
importances.to_csv(MODEL_DIR / "feature_importances.csv", index=False)

2020 {'model': 'extra_trees', 'validation_year': 2020, 'micro_auc': 0.8355078082139065, 'train_cases': 1819, 'valid_cases': 366} Elapsed seconds: 62.5


2021 {'model': 'extra_trees', 'validation_year': 2021, 'micro_auc': 0.8849253555418859, 'train_cases': 2185, 'valid_cases': 365} Elapsed seconds: 89.3


2022 {'model': 'extra_trees', 'validation_year': 2022, 'micro_auc': 0.914650949398587, 'train_cases': 2550, 'valid_cases': 365} Elapsed seconds: 120.1


                 mean       std       min       max
model                                              
extra_trees  0.878361  0.039978  0.835508  0.914651


## Fit all training cases and export predictions

The CSV file is a candidate, not automatically submitted to Kaggle. Shared evaluation files are updated by replacing only this model's earlier rows, so results from other notebooks are kept. The OOF files use the Kaggle wide format, like the other models.

In [5]:
def to_submission(predictions, run_nos):
    ordered = predictions.reshape(-1, H, len(UNITS)).transpose(0, 2, 1).reshape(-1, len(label_cols))
    frame = pd.DataFrame(ordered, columns=label_cols)
    frame.insert(0, "Run No", run_nos)
    frame = frame.set_index("Run No").loc[sample["Run No"]].reset_index()
    assert frame.columns.equals(sample.columns) and frame.shape == sample.shape
    assert np.isfinite(frame[label_cols].to_numpy()).all()
    return frame

def upsert(path, new_rows, same_run):
    """Replace rows from an earlier run of this experiment and keep everything else."""
    if path.exists():
        old = pd.read_csv(path)
        new_rows = pd.concat([old[~same_run(old)], new_rows], ignore_index=True)
    new_rows.to_csv(path, index=False)

train_indices = cases.index[cases["split"] == "train"].to_numpy()
test_indices = cases.index[cases["split"] == "test"].to_numpy()
xt, xe = X.iloc[rows_for(train_indices)], X.iloc[rows_for(test_indices)]
yt = Y.reshape(-1, len(UNITS))
test_prediction, full_importance = fit_predict_extra_trees(xt, yt, xe)
to_submission(test_prediction, cases.loc[test_indices, "Run No"].to_numpy()).to_csv(MODEL_DIR / f"submission_{MODEL}.csv", index=False)
full_importance.to_csv(MODEL_DIR / "feature_importances_full.csv", index=False)

CHANGE = "Initial Extra Trees, sklearn defaults, same features and folds as logistic regression"
upsert(EVAL_DIR / "cv_scores.csv", scores, lambda d: d["model"] == MODEL)
log = scores.copy()
log.insert(0, "date_utc", datetime.now(timezone.utc).isoformat())
log.insert(1, "who", "Group 71")
log["change"] = CHANGE
log["kaggle_score"] = np.nan
upsert(EVAL_DIR / "experiment_log.csv", log, lambda d: (d["model"] == MODEL) & (d["change"] == CHANGE))
(MODEL_DIR / "config.json").write_text(json.dumps({"seed": SEED, "validation_years": VALIDATION_YEARS, "gap_days": 7,
    "sklearn_version": sklearn.__version__, "params": {**ET_PARAMS, "n_jobs": 4, "random_state": SEED},
    "features": {u: columns_for(u) for u in UNITS}}, indent=2))
print("Artifacts written to", MODEL_DIR)

Artifacts written to C:\dev\TDT4173\TDT4173-Group-71\outputs\extra_trees


## Interpretation and comparison

The comparison uses the same validation splits as the other models in `cv_scores.csv`. Standard deviation is descriptive across three years, not a confidence interval. Importances are averaged over the three folds and normalized within each generator, so they sum to 1 per generator.

In [6]:
all_scores = pd.read_csv(EVAL_DIR / "cv_scores.csv")
comparison = all_scores.pivot(index="model", columns="validation_year", values="micro_auc")
comparison["mean"], comparison["std"] = comparison.mean(axis=1), comparison.std(axis=1)
print(comparison.round(4))

generic = importances.assign(feature=importances["feature"].str.replace(r"_(" + "|".join(map(re.escape, UNITS)) + r")$", "", regex=True))
generic.groupby("feature")["importance"].mean().sort_values(ascending=False).round(4)

validation_year        2020    2021    2022    mean     std
model                                                      
baseline             0.5693  0.5132  0.5731  0.5519  0.0335
extra_trees          0.8355  0.8849  0.9147  0.8784  0.0400
logistic_regression  0.8467  0.9046  0.9432  0.8982  0.0486


feature
wv                  0.1831
spread              0.1754
spread_t            0.1105
price_week_mean     0.0846
fill                0.0823
doy_cos             0.0599
doy_sin             0.0570
price               0.0486
inflow              0.0473
rank_fill           0.0320
season              0.0267
t                   0.0253
price_minus_mean    0.0223
weekday             0.0199
price_rank          0.0178
hour                0.0073
Name: importance, dtype: float64

Next step: train LightGBM on exactly the same features and folds, then compare the three models in `cv_scores.csv`. Test tuning and feature changes one at a time later, on the same folds. This notebook supplies modeling and importance interpretation; it does not replace the required EDA or the complete project report.